# 04 - Pipeline walkthrough (cached claims)

Three cached claims are run through every stage of the v1 pipeline with
`CACHE_ONLY=true`. No live news request is made.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
import config
config.CACHE_ONLY = True
from app.claim_analysis import analyze
from app.retrieval import retrieve
from app.ranking import rank_articles
from app.nli import classify
from app.decision import build_evidence, decide
from app.credibility import load_credibility

claims = [
    "South African human rights lawyer Navi Pillay won the 2026 Nobel Peace Prize.",
    "Manchester United beat Como 4-1 in a Champions League match on October 21, 2026.",
    "An anonymous source says a famous Bollywood actor will announce retirement next week.",
]
table = load_credibility()

## Stage 1: claim analysis

spaCy extracts entities and builds search queries.

In [2]:
for claim in claims:
    a = analyze(claim)
    print("=" * 80)
    print("CLAIM:", claim)
    print("entities:", a.entities)
    print("queries:", a.queries)

CLAIM: South African human rights lawyer Navi Pillay won the 2026 Nobel Peace Prize.
entities: [{'text': 'South African', 'label': 'NORP'}, {'text': 'Navi Pillay', 'label': 'PERSON'}, {'text': '2026', 'label': 'DATE'}]
queries: ['South African human rights lawyer Navi Pillay won the 2026 Nobel Peace', 'Navi Pillay South African 2026', 'Navi Pillay South African won']
CLAIM: Manchester United beat Como 4-1 in a Champions League match on October 21, 2026.
entities: [{'text': 'Manchester United', 'label': 'PERSON'}, {'text': 'Como 4-1', 'label': 'PRODUCT'}, {'text': 'Champions League', 'label': 'ORG'}, {'text': 'October 21, 2026', 'label': 'DATE'}]
queries: ['Manchester United beat Como 4-1 a Champions League match October 21 2026', 'Champions League Manchester United Como 4-1 October 21 2026', 'Champions League Manchester United beat']
CLAIM: An anonymous source says a famous Bollywood actor will announce retirement next week.
entities: [{'text': 'Bollywood', 'label': 'ORG'}, {'text': 'n

## Stage 2: retrieval (cache only)

Articles come from the local cache. No network call is made.

In [3]:
for claim in claims:
    a = analyze(claim)
    arts = retrieve(a.queries, cache_only=True)
    print("=" * 80)
    print("CLAIM:", claim)
    print("retrieved articles:", len(arts))
    for x in arts[:5]:
        print(" -", x.source_domain, "|", x.title[:70])

CLAIM: South African human rights lawyer Navi Pillay won the 2026 Nobel Peace Prize.
retrieved articles: 36
 - npr.org | South African Human Rights lawyer Navi Pillay wins Nobel Peace Prize
 - salon.com | Trump: I should have 8 Nobel Peace Prizes
 - pbs.org | Watch Navi Pillay Win Nobel Peace Prize for Upholding Global Justice
 - newrepublic.com | Nobel Peace Prize Winner Mocks Trump Immediately After Accepting Award
 - punchng.com | South African judge Navi Pillay wins 2026 Nobel Peace Prize


CLAIM: Manchester United beat Como 4-1 in a Champions League match on October 21, 2026.
retrieved articles: 41
 - worldsoccertalk.com | How to watch Liverpool vs Villarreal match in the USA: Live stream and
 - worldsoccertalk.com | How to watch Porto vs PSV match in the USA: Live stream and TV for 202
 - worldsoccertalk.com | How to watch PSG vs Barcelona match in the USA: Live stream and TV for
 - worldsoccertalk.com | How to watch Manchester City vs AEK Athens match in the USA: Live stre
 - worldsoccertalk.com | How to watch Real Madrid vs RB Leipzig match in the USA: Live stream a


CLAIM: An anonymous source says a famous Bollywood actor will announce retirement next week.
retrieved articles: 20
 - deadline.com | ‘The History Of Concrete’ Cracks $1 Million, ‘NAZA’ Lifts Off – Specia
 - deadline.com | The “Algorithm Breaking” ‘Hunkkaar’ Has Challenged Indian Audiences On
 - mubi.com | Rushes | Netflix Fires Chief Content Officer, Soderbergh Takes Over “T
 - economictimes.indiatimes.com | Jeffrey Archer was Brit, but dil tha Hindustani
 - timesofindia.indiatimes.com | Trump says he deserves 8 Nobel Peace Prizes, calls award ‘forever disc


## Stage 3: ranking

Articles are scored by claim similarity and recency.

In [4]:
for claim in claims:
    a = analyze(claim)
    arts = retrieve(a.queries, cache_only=True)
    ranked = rank_articles(a.text, arts)
    print("=" * 80)
    print("CLAIM:", claim)
    print("ranked articles:", len(ranked))
    for r in ranked[:5]:
        print(f"  sim={r.similarity:.3f} recency={r.recency_factor:.2f} score={r.score:.3f} {r.source_domain}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

CLAIM: South African human rights lawyer Navi Pillay won the 2026 Nobel Peace Prize.
ranked articles: 6
  sim=0.974 recency=0.60 score=0.584 npr.org
  sim=0.948 recency=0.60 score=0.569 thejournal.ie
  sim=0.925 recency=0.60 score=0.555 economictimes.indiatimes.com
  sim=0.915 recency=0.60 score=0.549 aljazeera.com
  sim=0.903 recency=0.60 score=0.542 radaronline.com


CLAIM: Manchester United beat Como 4-1 in a Champions League match on October 21, 2026.
ranked articles: 6
  sim=0.849 recency=0.60 score=0.509 football-italia.net
  sim=0.614 recency=0.60 score=0.368 the4thofficial.net
  sim=0.613 recency=0.60 score=0.368 aljazeera.com
  sim=0.596 recency=0.60 score=0.358 worldsoccertalk.com
  sim=0.574 recency=0.60 score=0.344 skysports.com


CLAIM: An anonymous source says a famous Bollywood actor will announce retirement next week.
ranked articles: 1
  sim=0.403 recency=0.60 score=0.242 economictimes.indiatimes.com


## Stage 4: NLI

Each article snippet is scored against the claim.

In [5]:
for claim in claims:
    a = analyze(claim)
    arts = retrieve(a.queries, cache_only=True)
    ranked = rank_articles(a.text, arts)
    snippets = [r.snippet for r in ranked]
    res = classify(a.text, snippets) if snippets else []
    print("=" * 80)
    print("CLAIM:", claim)
    for r, n in zip(ranked, res):
        print(f"  {r.source_domain:28s} {n.stance:8s} entail={n.p_entail:.3f} neutral={n.p_neutral:.3f} contra={n.p_contra:.3f}")

CLAIM: South African human rights lawyer Navi Pillay won the 2026 Nobel Peace Prize.
  npr.org                      SUPPORTS entail=0.997 neutral=0.003 contra=0.001
  thejournal.ie                SUPPORTS entail=0.997 neutral=0.002 contra=0.001
  economictimes.indiatimes.com SUPPORTS entail=0.995 neutral=0.005 contra=0.001
  aljazeera.com                NEUTRAL  entail=0.162 neutral=0.837 contra=0.001
  radaronline.com              SUPPORTS entail=0.959 neutral=0.037 contra=0.003
  punchng.com                  SUPPORTS entail=0.899 neutral=0.099 contra=0.002


CLAIM: Manchester United beat Como 4-1 in a Champions League match on October 21, 2026.
  football-italia.net          REFUTES  entail=0.000 neutral=0.001 contra=0.999
  the4thofficial.net           NEUTRAL  entail=0.002 neutral=0.912 contra=0.086
  aljazeera.com                REFUTES  entail=0.000 neutral=0.003 contra=0.997
  worldsoccertalk.com          NEUTRAL  entail=0.001 neutral=0.996 contra=0.003
  skysports.com                REFUTES  entail=0.001 neutral=0.011 contra=0.988
  bbc.co.uk                    REFUTES  entail=0.000 neutral=0.028 contra=0.971


CLAIM: An anonymous source says a famous Bollywood actor will announce retirement next week.
  economictimes.indiatimes.com REFUTES  entail=0.000 neutral=0.000 contra=1.000


## Stage 5: credibility and decision

Credibility weights are applied and the safety-gated decision runs.

In [6]:
for claim in claims:
    a = analyze(claim)
    arts = retrieve(a.queries, cache_only=True)
    ranked = rank_articles(a.text, arts)
    snippets = [r.snippet for r in ranked]
    res = classify(a.text, snippets) if snippets else []
    ev = build_evidence(ranked, res, credibility_table=table)
    v = decide(a.text, ev, queries=a.queries, latency_ms=0)
    print("=" * 80)
    print("CLAIM:", claim)
    print("VERDICT:", v.verdict)
    print("support:", round(v.support_score, 3), "refute:", round(v.refute_score, 3),
          "total:", round(v.total_weight, 3), "confidence:", round(v.confidence, 3))
    print("explanation:", v.explanation)
    print("top evidence:")
    for e in v.evidence[:3]:
        print(f"  {e.source_domain:28s} {e.stance:8s} sim={e.similarity:.3f} cred={e.credibility:.2f}")

CLAIM: South African human rights lawyer Navi Pillay won the 2026 Nobel Peace Prize.
VERDICT: SUPPORTED
support: 2.663 refute: 0 total: 2.722 confidence: 0.978
explanation: SUPPORTED: 5 relevant sources support the claim (top: npr.org, economictimes.indiatimes.com); support 2.663 vs refute 0.000; 1 relevant neutral article(s) treated as no information.
top evidence:
  npr.org                      SUPPORTS sim=0.974 cred=1.00
  thejournal.ie                SUPPORTS sim=0.948 cred=0.40
  economictimes.indiatimes.com SUPPORTS sim=0.925 cred=0.70


CLAIM: Manchester United beat Como 4-1 in a Champions League match on October 21, 2026.
VERDICT: REFUTED
support: 0 refute: 0.994 total: 0.999 confidence: 0.995
explanation: REFUTED: 3 relevant sources refute the claim (top: aljazeera.com, football-italia.net); support 0.000 vs refute 0.994; 2 relevant neutral article(s) treated as no information.
top evidence:
  football-italia.net          REFUTES  sim=0.849 cred=0.40
  the4thofficial.net           NEUTRAL  sim=0.614 cred=0.40
  aljazeera.com                REFUTES  sim=0.613 cred=0.70


CLAIM: An anonymous source says a famous Bollywood actor will announce retirement next week.
VERDICT: UNVERIFIED
support: 0.0 refute: 0.0 total: 0.0 confidence: 0.0
explanation: UNVERIFIED: only 1 kept article(s); at least 2 are required.
top evidence:
  economictimes.indiatimes.com REFUTES  sim=0.403 cred=0.70


## Summary

The three claims give SUPPORTED, REFUTED and UNVERIFIED respectively, showing
the abstention behaviour on weak evidence.

In [7]:
print("notebook 04 complete")

notebook 04 complete
